In [ ]:

from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

In [ ]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [ ]:
class CombinedModeWeightNet(nn.Module):
    def __init__(self, mode_model, weight_model):
        super().__init__()
        self.mode_model = mode_model
        self.weight_model = weight_model

    def forward(self, x_img, x_cond):
        mode = self.mode_model(x_img, x_cond)   # [B, 1]
        weight = self.weight_model(x_img, x_cond)  # [B, 1]
        return torch.cat((mode, weight), dim=1)   # [B, 2]

from only_mode_only_weight_v3 import No_normal_modewieght_net

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode0_model = No_normal_modewieght_net().to(device)
weight0_model = No_normal_modewieght_net().to(device)

mode0_model.load_state_dict(torch.load('models/only_first_mode_no_normalization_with_less_dropout.pth', map_location=torch.device(device)))
weight0_model.load_state_dict(torch.load('models/only_first_weight_no_normalization_with_less_dropout_100.pth', map_location=torch.device(device)))

cnn = CombinedModeWeightNet(mode0_model, weight0_model).to(device)
cnn.eval()

In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        '''
        standard ResNet style convolutional block, for image processing
        '''
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            # this adds on correct residual in case channels have increased
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2

In [ ]:
class UnetDown(nn.Module):
    """
    Downsampling path for U-Net, reduces spatial resolution while increasing feature depth
    Input: Image batch, size (batchsize, 1, 32, 32)
    Output: size (batchsize, out_channels, 16, 16)
    Output:
    """
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        '''
        process and downscale the image feature maps
        '''
        layers = [ResidualConvBlock(
            in_channels, out_channels), nn.MaxPool2d(2)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Doubles spatial dimensions, halves feature dimensions
        # My channel dimension for image will always be 1, greyscale
        return self.model(x)


In [ ]:
class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        '''
        process and upscale the image feature maps
        Doubles spatial size, but decreases channels:
        input: 2 vectors of size (binsize, in_channels / 2, h, w) 
        output: (binsize, outchannels, 2h, 2w)
        '''
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        """
        x is the upsampled features from previous decoder layer
        skip is the skip connection from the encoder, same size as x
        """
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x

In [ ]:
class EmbedFC(nn.Module):
    """
    Use FC layer for embedding 1-d metadata, like modes+weights
    (putting into higher dimension)
    Effectively our conditional
    input: Conditional, size (batchsize, input_dim = 4+4)
    Output: Higherdimensional tensor, size (batchsize, output_dim)
    
    """
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things  
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assumes you already have these:
# - ResidualConvBlock(in_ch, out_ch, is_res=True)
# - UnetDown(in_ch, out_ch)
# - UnetUp(in_ch, out_ch)
# - EmbedFC(in_dim, out_dim)

class SeqEncoder1D(nn.Module):
    """
    Encodes the dispersion sequence (concatenated [mode, weight, params]) of shape [B, W, 6]
    into a fixed-length embedding via Conv1d + global pooling.
    """
    def __init__(self, in_ch=6, hid=128, out=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, hid, kernel_size=5, padding=2),
            nn.GroupNorm(8, hid),
            nn.GELU(),
            nn.Conv1d(hid, hid, kernel_size=5, padding=2),
            nn.GroupNorm(8, hid),
            nn.GELU(),
            nn.Conv1d(hid, out, kernel_size=5, padding=2),
            nn.GELU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(out * 2, out),
            nn.GELU(),
            nn.Linear(out, out),
        )

    def forward(self, x_bw6):  # [B, W, 6] or [B, 6, W]
        # ensure channels-first for Conv1d: [B, 6, W]
        if x_bw6.dim() != 3:
            raise ValueError(f"SeqEncoder1D expects [B, W, 6] or [B, 6, W], got {tuple(x_bw6.shape)}")
        if x_bw6.shape[-1] == 6:   # [B, W, 6] -> [B, 6, W]
            x = x_bw6.transpose(1, 2).contiguous()
        elif x_bw6.shape[1] == 6:  # already [B, 6, W]
            x = x_bw6
        else:
            raise ValueError(f"SeqEncoder1D needs 6 feature channels; got {tuple(x_bw6.shape)}")

        h = self.net(x)           # [B, out, W]
        h_mean = h.mean(dim=-1)   # [B, out]
        h_max  = h.amax(dim=-1)   # [B, out]
        z = torch.cat([h_mean, h_max], dim=-1)  # [B, 2*out]
        return self.proj(z)       # [B, out]


class ContextUnet(nn.Module):
    """
    U-Net for conditional image generation (32x32), now conditioned on the FULL dispersion curve.

    Conditioning inputs (per sample):
      - mw_seq    : [B, W, 2]    -> columns [mode, weight]
      - param_seq : [B, W, 4]    -> columns [λ, lattice_norm, n_atom, n_substrate]
      (W is typically 100, but any W works.)

    Pipeline:
      [B, W, 2], [B, W, 4] -> concat -> [B, W, 6] -> 1D conv encoder -> [B, cond_embed]
      -> Linear -> cond_channels scalars -> tile to [B, cond_channels, 32, 32]
      -> concatenate to image input channels, rest of U-Net unchanged.

    Timestep t is embedded via an MLP and injected on the up path (same as before).
    """

    def __init__(self, in_channels=1, n_feat=256, use_time_embed=True,
                 cond_channels=32, cond_embed_dim=256):
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.use_time_embed = use_time_embed
        self.cond_channels = cond_channels

        # Sequence encoder for dispersion curve
        self.seq_encoder = SeqEncoder1D(in_ch=6, hid=128, out=cond_embed_dim)
        self.cond_head   = nn.Linear(cond_embed_dim, cond_channels)

        lifted_in = in_channels + cond_channels

        # Encoder
        self.init_conv = ResidualConvBlock(lifted_in, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)          # 32x32 -> 16x16
        self.down2 = UnetDown(n_feat, 2 * n_feat)      # 16x16 -> 8x8

        # Latent pooling to 1x1
        self.to_vec = nn.Sequential(nn.AvgPool2d(8), nn.GELU())

        # Optional timestep embedding (kept as MLP)
        if use_time_embed:
            self.timeembed1 = EmbedFC(1, 2 * n_feat)
            self.timeembed2 = EmbedFC(1, 1 * n_feat)

        # Decoder
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8),  # 1x1 -> 8x8
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        self.up1 = UnetUp(4 * n_feat, n_feat)          # (up + skip) 8x8 -> 16x16
        self.up2 = UnetUp(2 * n_feat, n_feat)          # 16x16 -> 32x32

        # Output head
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    @staticmethod
    def _build_cond_maps_from_seq(mw_seq, param_seq, H, W, cond_head, seq_encoder):
        """
        mw_seq    : [B, W, 2]
        param_seq : [B, W, 4]
        Returns tiled cond maps: [B, Cc, H, W], where Cc = cond_channels
        """
        if mw_seq.dim() != 3 or param_seq.dim() != 3:
            raise ValueError(f"Expected mw_seq [B,W,2] and param_seq [B,W,4], got "
                             f"{tuple(mw_seq.shape)} and {tuple(param_seq.shape)}")
        if mw_seq.size(-1) != 2 or param_seq.size(-1) != 4:
            raise ValueError("mw_seq last dim must be 2; param_seq last dim must be 4.")
        if mw_seq.size(0) != param_seq.size(0) or mw_seq.size(1) != param_seq.size(1):
            raise ValueError("mw_seq and param_seq must agree on batch and W.")

        seq = torch.cat([mw_seq, param_seq], dim=-1)   # [B, W, 6]
        z = seq_encoder(seq)                           # [B, cond_embed_dim]
        cond_vec = cond_head(z)                        # [B, cond_channels]
        cond_maps = cond_vec.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)  # [B,Cc,H,W]
        return cond_maps

    def forward(self, x, mw_seq, param_seq, t):
        """
        x        : [B, 1, 32, 32]           (noisy waveguide / latent; your dataset quarters upstream)
        mw_seq   : [B, W, 2]                (mode, weight sequences)
        param_seq: [B, W, 4]                (λ, lattice_norm, n_atom, n_substrate)
        t        : [B, 1]                   (scalar timestep per sample)
        """
        B, _, H, W = x.shape
        if (H, W) != (32, 32):
            raise ValueError(f"This U-Net assumes 32x32; got {(H, W)}")

        # Build tiled condition maps from the dispersion sequence
        cond_maps = self._build_cond_maps_from_seq(
            mw_seq, param_seq, H, W, self.cond_head, self.seq_encoder
        )  # [B, cond_channels, 32, 32]

        # Concatenate to input
        x_in = torch.cat([x, cond_maps], dim=1)        # [B, 1+cond_channels, 32, 32]

        # Encoder
        x0 = self.init_conv(x_in)    # [B, n_feat, 32, 32]
        d1 = self.down1(x0)          # [B, n_feat, 16, 16]
        d2 = self.down2(d1)          # [B, 2*n_feat, 8, 8]
        h  = self.to_vec(d2)         # [B, 2*n_feat, 1, 1]

        # Decoder (+ optional time embeddings)
        up1 = self.up0(h)            # [B, 2*n_feat, 8, 8]

        if self.use_time_embed:
            temb1 = self.timeembed1(t).view(B, 2 * self.n_feat, 1, 1)
            temb2 = self.timeembed2(t).view(B, 1 * self.n_feat, 1, 1)
            up1 = up1 + temb1

        u2 = self.up1(up1, d2)       # -> [B, n_feat, 16, 16]
        if self.use_time_embed:
            u2 = u2 + temb2

        u3 = self.up2(u2, d1)        # -> [B, n_feat, 32, 32]

        out = self.out(torch.cat([u3, x0], dim=1))  # [B, 1, 32, 32]
        return out


In [ ]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [ ]:
import torch
import torch.nn as nn
import numpy as np

# assumes you already have:
# ddpm_schedules(beta1, beta2, n_T) -> dict of schedules with keys:
#   "beta_t", "alpha_t", "alpha_t_bar", plus convenience terms used below
# If your names differ, adapt the buffer names in __init__ and forward/sample.

class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        """
        nn_model: a U-Net with signature
                  nn_model(x_t, cond_seq, params_seq, t_norm) -> predicted noise, shape [B,1,32,32]
        betas   : (beta1, beta2) range for linear schedule (or however your ddpm_schedules uses it)
        n_T     : number of diffusion steps
        device  : torch.device
        drop_prob: classifier-free dropout probability (probability of dropping conditioning)
        """
        super().__init__()
        self.nn_model = nn_model.to(device)

        # Register schedule buffers
        sched = ddpm_schedules(betas[0], betas[1], n_T)
        # Expect the following keys; adapt if yours differ.
        # sqrtab[t]        = sqrt(alpha_bar_t)
        # sqrtmab[t]       = sqrt(1 - alpha_bar_t)
        # oneover_sqrta[t] = 1 / sqrt(alpha_t)
        # mab_over_sqrtmab[t] = (1 - alpha_t) / sqrt(1 - alpha_bar_t)
        # sqrt_beta_t[t]   = sqrt(beta_t)
        for k, v in sched.items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    @staticmethod
    def _apply_mask_seq(cond_seq, params_seq, context_mask):
        """
        Classifier-free dropout on the whole dispersion sequence.

        Inputs
        ------
        cond_seq   : [B, W, 2]
        params_seq : [B, W, 4]
        context_mask: [B] or [B,1] or [B,1,1] of {0,1}; 1 => DROP conditioning

        Returns
        -------
        masked_cond_seq   : [B, W, 2]
        masked_params_seq : [B, W, 4]
        """
        if context_mask.dim() == 1:
            context_mask = context_mask.unsqueeze(1)   # [B,1]
        if context_mask.dim() == 2:
            context_mask = context_mask.unsqueeze(2)   # [B,1,1]
        # Broadcast to [B,W,*]
        mask = context_mask  # [B,1,1]
        if cond_seq.dim() != 3 or params_seq.dim() != 3:
            raise ValueError(f"Expected cond_seq [B,W,2] and params_seq [B,W,4]; "
                             f"got {tuple(cond_seq.shape)} and {tuple(params_seq.shape)}")
        if cond_seq.size(-1) != 2 or params_seq.size(-1) != 4:
            raise ValueError("cond_seq last dim must be 2; params_seq last dim must be 4.")

        masked_cond = cond_seq * (1.0 - mask)          # [B,W,2]
        masked_params = params_seq * (1.0 - mask)      # [B,W,4]
        return masked_cond, masked_params

    def forward(self, x0, cond_seq, params_seq):
        """
        One training step.

        x0         : [B,1,32,32]  clean quartered pattern (target)
        cond_seq   : [B,W,2]      (mode, weight) per wavelength
        params_seq : [B,W,4]      (λ, lattice_norm, n_atom, n_substrate)

        Returns scalar MSE loss between true noise and predicted noise.
        """
        B = x0.shape[0]
        if x0.shape[1:] != (1, 32, 32):
            raise ValueError(f"x0 must be [B,1,32,32]; got {tuple(x0.shape)}")

        # Random timesteps
        _ts = torch.randint(1, self.n_T + 1, (B,), device=self.device)  # [B]
        noise = torch.randn_like(x0)

        # q(x_t | x_0)
        x_t = (
            self.sqrtab[_ts, None, None, None] * x0
            + self.sqrtmab[_ts, None, None, None] * noise
        )

        # Classifier-free dropout on the sequences
        context_mask = torch.bernoulli(
            torch.full((B,), self.drop_prob, device=self.device)
        )  # [B] with {0,1}
        cond_masked, params_masked = self._apply_mask_seq(cond_seq, params_seq, context_mask)

        # Normalized timestep in [0,1]
        t_norm = (_ts.float() / self.n_T).unsqueeze(1)  # [B,1]

        # Predict noise
        pred_noise = self.nn_model(x_t, cond_masked, params_masked, t_norm)  # [B,1,32,32]
        return self.loss_mse(noise, pred_noise)

    @torch.no_grad()
    def sample(self, n_sample, size, device, cond_seq, params_seq, guide_w=0.0):
        """
        Ancestral sampling with classifier-free guidance (CFG).

        Args
        ----
        n_sample : int
        size     : tuple, e.g. (1, 32, 32)
        device   : torch.device
        cond_seq : [n_sample, W, 2]
        params_seq: [n_sample, W, 4]
        guide_w  : float, guidance scale (0 => no CFG)

        Returns
        -------
        x_0 tensor [n_sample, 1, 32, 32], and numpy trajectory for visualization.
        """
        assert cond_seq.shape[0] == n_sample and params_seq.shape[0] == n_sample
        if size != (1, 32, 32):
            raise ValueError(f"size must be (1,32,32); got {size}")

        x_i = torch.randn(n_sample, *size, device=device)

        # Build masked/unmasked contexts
        context_mask_cond   = torch.zeros(n_sample, device=device)  # keep
        context_mask_uncond = torch.ones(n_sample,  device=device)  # drop

        cond_c,  par_c  = self._apply_mask_seq(cond_seq,  params_seq, context_mask_cond)
        cond_uc, par_uc = self._apply_mask_seq(cond_seq,  params_seq, context_mask_uncond)

        x_i_store = []
        for i in range(self.n_T, 0, -1):
            t_norm = torch.full((n_sample, 1), i / self.n_T, device=device)

            # Double batch for CFG
            x_in = torch.cat([x_i, x_i], dim=0)               # [2B,1,32,32]
            t_in = torch.cat([t_norm, t_norm], dim=0)         # [2B,1]
            cond_in = torch.cat([cond_c, cond_uc], dim=0)     # [2B,W,2]
            par_in  = torch.cat([par_c,  par_uc],  dim=0)     # [2B,W,4]

            eps = self.nn_model(x_in, cond_in, par_in, t_in)  # [2B,1,32,32]
            eps1, eps2 = eps[:n_sample], eps[n_sample:]

            # CFG combine
            eps_cfg = (1 + guide_w) * eps1 - guide_w * eps2

            z = torch.randn_like(x_i) if i > 1 else 0.0
            x_i = (
                self.oneover_sqrta[i] * (x_i - eps_cfg * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )

            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        x_i_store = np.array(x_i_store)
        return x_i, x_i_store

In [ ]:
import importlib
import waveguide_dataset_dispersion
importlib.reload(waveguide_dataset_dispersion)
from waveguide_dataset_dispersion import WaveguideDatasetPaired

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt  # needed for plotting

def _ensure_sequences(cond_or_top: torch.Tensor,
                      params: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Normalize inputs to sequences for the new DDPM:

    Inputs:
      cond_or_top:
        - [B, W, 2]  (new dataset: (mode, weight) per wavelength)  -> keep
        - [B, 2]     (old: single top pair)                        -> tile to W
        - [B, 4, 2]  (old: 4 pairs)                                 -> treat W=4

      params:
        - [B, W, 4]  (new dataset: per-λ params) -> keep
        - [B, 4]     (old: single param set)     -> tile to W

    Returns:
      cond_seq   : [B, W, 2]
      params_seq : [B, W, 4]
    """
    # Determine cond_seq
    if cond_or_top.dim() == 3 and cond_or_top.size(-1) == 2:
        # Either [B,W,2] (new) or [B,4,2] (old-fixed-W)
        cond_seq = cond_or_top
        W_from_cond = cond_seq.size(1)
    elif cond_or_top.dim() == 2 and cond_or_top.size(1) == 2:
        # Single top pair -> will tile after we know W
        cond_seq = None
        W_from_cond = None
    else:
        raise ValueError(f"cond/top must be [B,W,2], [B,4,2], or [B,2]; got {tuple(cond_or_top.shape)}")

    # Determine params_seq and W
    if params.dim() == 3 and params.size(-1) == 4:
        params_seq = params
        W_from_params = params_seq.size(1)
    elif params.dim() == 2 and params.size(1) == 4:
        params_seq = None
        W_from_params = None
    else:
        raise ValueError(f"params must be [B,W,4] or [B,4]; got {tuple(params.shape)}")

    # Resolve W
    if cond_seq is not None and params_seq is not None:
        if W_from_cond != W_from_params:
            raise ValueError(f"W mismatch: cond W={W_from_cond}, params W={W_from_params}")
        W = W_from_cond
    elif cond_seq is not None:
        W = W_from_cond
    elif params_seq is not None:
        W = W_from_params
    else:
        raise ValueError("Cannot infer W: both cond and params are non-sequence. Provide one as [B,W,*].")

    B = cond_or_top.size(0)
    # Tile if needed
    if cond_seq is None:
        # [B,2] -> [B,W,2]
        cond_seq = cond_or_top.unsqueeze(1).expand(B, W, 2).contiguous()
    if params_seq is None:
        # [B,4] -> [B,W,4]
        params_seq = params.unsqueeze(1).expand(B, W, 4).contiguous()

    return cond_seq, params_seq


@torch.no_grad()
def compare_waveguides_3gens(
    ddpm, test_loader, device, guide_w=2.0, n_rows=12,
    save_path=None, binarize=False, thresh=0.5
):
    """
    For each of the first n_rows items from the test loader, generate 3 waveguides
    conditioned on the same dispersion sequence (cond_seq, params_seq), and display as:
        Real (red) | Gen 1 (black) | Gen 2 (black) | Gen 3 (black)

    test_loader should yield either:
      NEW: (cond:[B,W,2], params:[B,W,4], x_real:[B,1,32,32])
      OLD: (top_pair:[B,2] or [B,4,2], params:[B,4], x_real:[B,1,32,32])
           (old inputs are automatically tiled across W)
    """
    ddpm.eval()

    # Grab one batch
    try:
        cond_or_top, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    # Trim to n_rows
    n = min(n_rows, cond_or_top.shape[0])
    cond_or_top = cond_or_top[:n].to(device)
    params      = params[:n].to(device)
    x_real      = x_real[:n].to(device)

    # Ensure sequences
    cond_seq, params_seq = _ensure_sequences(cond_or_top, params)  # [n,W,2], [n,W,4]
    W = cond_seq.size(1)

    # Repeat each condition 3× (independent noise makes different samples)
    num_gens = 3
    cond_seq_rep   = cond_seq.repeat_interleave(num_gens, dim=0)     # [n*3,W,2]
    params_seq_rep = params_seq.repeat_interleave(num_gens, dim=0)   # [n*3,W,4]

    # Sample all at once
    x_gen_all, _ = ddpm.sample(
        n_sample=n * num_gens,
        size=(1, 32, 32),
        device=device,
        cond_seq=cond_seq_rep,
        params_seq=params_seq_rep,
        guide_w=guide_w,
    )  # [n*3,1,32,32]

    # Reshape to [n,3,1,32,32]
    x_gen_all = x_gen_all.view(n, num_gens, 1, 32, 32)

    # To CPU numpy
    real_np = x_real.detach().cpu().numpy()           # [n,1,32,32]
    gen_np  = x_gen_all.detach().cpu().numpy()        # [n,3,1,32,32]

    # Clamp and optional binarization
    real_np = np.clip(real_np, 0.0, 1.0)
    gen_np  = np.clip(gen_np,  0.0, 1.0)
    if binarize:
        real_np = (real_np >= thresh).astype(np.float32)
        gen_np  = (gen_np  >= thresh).astype(np.float32)

    # Plot: 4 columns => Real + 3 gens
    cols = 4
    fig_h = max(2, n * 1.1)
    fig, axs = plt.subplots(n, cols, figsize=(cols * 2.2, fig_h), squeeze=False)

    for i in range(n):
        # Real (red)
        ax = axs[i, 0]
        ax.imshow(real_np[i, 0], cmap="Reds", vmin=0, vmax=1)
        if i == 0: ax.set_title("Real", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # Gen 1..3 (black). Use 1 - gen for black-on-white with 'binary_r' cmap
        for j in range(num_gens):
            ax = axs[i, j + 1]
            ax.imshow(1.0 - gen_np[i, j, 0], cmap="binary_r", vmin=0, vmax=1)
            if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

def train_waveguide_ddpm(
    h5_path='wavelength_grid_dataset.h5',
    stats_path="waveguide_stats_log_norm_above90.npz",
    save_dir="./data/diffusion_params_v5_topmode_nonorm/",
    n_epoch=20,
    batch_size=256,
    n_T=400,
    n_feat=128,
    lrate=1e-4,
    drop_prob=0.1,
    betas=(1e-4, 0.02),
    ws_test=(0.0, 0.5, 2.0),
    save_model=True,
    test_eval_fraction=0.25,   # portion of test loader to evaluate each epoch
    device=None,
    cnn=None,                  # optional evaluator CNN: (x:[B,1,32,32], p4:[B,4]) -> [B,2]
    gamma: float = 0.0,        # blend: gamma*noise_loss + (1-gamma)*cnn_loss
):
    """
    Trains a DDPM that is conditioned ONLY on dispersion sequences:
      - cond_seq   : [B, W, 2]  (mode, weight) per wavelength
      - params_seq : [B, W, 4]  (λ, lattice_norm, n_atom, n_substrate)

    If gamma < 1 and a CNN is provided, adds a consistency term comparing
    CNN(x_real, p4_peak) vs CNN(x0_hat, p4_peak), where p4_peak is taken
    at the wavelength with the highest weight for each sample.
    """
    assert 0.0 <= gamma <= 1.0, "gamma must be in [0,1]"
    if gamma > 0 and cnn is None:
        raise ValueError("gamma>0 requires a CNN (cnn=...)")

    os.makedirs(save_dir, exist_ok=True)
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")

    # ---- Data ----
    train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
    test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # ---- Model ----
    unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
    ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)
    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    # ---- Helpers ----
    def pick_p4_peak(cond_seq: torch.Tensor, params_seq: torch.Tensor) -> torch.Tensor:
        """
        cond_seq  : [B,W,2]  (mode, weight)
        params_seq: [B,W,4]
        returns p4_peak: [B,4] chosen at argmax weight per sample.
        """
        # weight column = 1
        idx = torch.argmax(cond_seq[..., 1], dim=1)                 # [B]
        idx_exp = idx[:, None, None].expand(-1, 1, 4)               # [B,1,4]
        p4 = params_seq.gather(dim=1, index=idx_exp).squeeze(1)     # [B,4]
        return p4

    # Prep CNN (fixed evaluator) if used
    use_cnn = (cnn is not None) and (gamma < 1.0)
    if use_cnn:
        cnn = cnn.to(device).eval()
        for p in cnn.parameters():
            p.requires_grad_(False)

    # ---- Tracking ----
    train_losses, test_losses = [], []
    best_test_loss = float("inf")

    def evaluate_on_test_portion():
        """Blended loss on a subset: gamma*noise + (1-gamma)*cnn, with dropout disabled."""
        ddpm.eval()
        was = ddpm.drop_prob
        ddpm.drop_prob = 0.0  # disable CFG dropout during eval

        total, count = 0.0, 0
        max_batches = max(1, int(np.ceil(test_eval_fraction * len(test_loader)))) if len(test_loader) > 0 else 1

        with torch.no_grad():
            for b_idx, (cond_seq, params_seq, x_real) in enumerate(test_loader):
                x_real     = x_real.to(device, non_blocking=True)        # [B,1,32,32]
                cond_seq   = cond_seq.to(device, non_blocking=True)      # [B,W,2]
                params_seq = params_seq.to(device, non_blocking=True)    # [B,W,4]

                # Noise loss (no dropout in eval)
                loss_noise = ddpm(x_real, cond_seq, params_seq)

                # CNN loss
                if use_cnn and gamma < 1.0:
                    B = x_real.size(0)
                    t = torch.randint(1, n_T + 1, (B,), device=device)
                    noise = torch.randn_like(x_real)
                    x_t = ddpm.sqrtab[t, None, None, None] * x_real + ddpm.sqrtmab[t, None, None, None] * noise
                    t_norm = (t.float() / n_T).unsqueeze(1)

                    pred_noise = ddpm.nn_model(x_t, cond_seq, params_seq, t_norm)
                    x0_hat = (x_t - ddpm.sqrtmab[t, None, None, None] * pred_noise) / ddpm.sqrtab[t, None, None, None]
                    x0_hat = x0_hat.clamp(0.0, 1.0)

                    # choose one param per sample for the evaluator CNN
                    p4_peak = pick_p4_peak(cond_seq, params_seq)                 # [B,4]
                    targ_top = cnn(x_real, p4_peak)                               # [B,2]
                    pred_top = cnn(x0_hat, p4_peak)                               # [B,2]
                    loss_cnn = F.mse_loss(pred_top, targ_top)
                else:
                    loss_cnn = x_real.new_zeros(())

                loss = gamma * loss_noise + (1.0 - gamma) * loss_cnn
                total += float(loss.item())
                count += 1
                if b_idx + 1 >= max_batches:
                    break

        ddpm.drop_prob = was
        return total / max(count, 1)

    # ---- Training loop ----
    for ep in range(n_epoch):
        print(f"epoch {ep}")
        ddpm.train()

        # linear LR decay
        for g in optim.param_groups:
            g["lr"] = lrate * (1 - ep / n_epoch)

        pbar = tqdm(train_loader)
        loss_ema = None
        train_loss_sum, train_loss_batches = 0.0, 0

        for cond_seq, params_seq, x_real in pbar:
            x_real     = x_real.to(device, non_blocking=True)       # [B,1,32,32]
            cond_seq   = cond_seq.to(device, non_blocking=True)     # [B,W,2]
            params_seq = params_seq.to(device, non_blocking=True)   # [B,W,4]

            # 1) Diffusion noise loss
            optim.zero_grad(set_to_none=True)
            loss_noise = ddpm(x_real, cond_seq, params_seq)

            # 2) Optional CNN consistency term
            if use_cnn and gamma < 1.0:
                B = x_real.size(0)
                t = torch.randint(1, n_T + 1, (B,), device=device)
                noise = torch.randn_like(x_real)
                x_t = ddpm.sqrtab[t, None, None, None] * x_real + ddpm.sqrtmab[t, None, None, None] * noise
                t_norm = (t.float() / n_T).unsqueeze(1)

                # classifier-free dropout on sequences (same drop_prob as model)
                context_mask = torch.bernoulli(torch.full((B,), ddpm.drop_prob, device=device))
                cond_m, params_m = ddpm._apply_mask_seq(cond_seq, params_seq, context_mask)

                pred_noise = ddpm.nn_model(x_t, cond_m, params_m, t_norm)
                x0_hat = (x_t - ddpm.sqrtmab[t, None, None, None] * pred_noise) / ddpm.sqrtab[t, None, None, None]
                x0_hat = x0_hat.clamp(0.0, 1.0)

                with torch.no_grad():
                    p4_peak = pick_p4_peak(cond_seq, params_seq)     # [B,4]
                    targ_top = cnn(x_real, p4_peak)                  # [B,2]
                pred_top = cnn(x0_hat, p4_peak)                      # [B,2]
                loss_cnn = F.mse_loss(pred_top, targ_top)
            else:
                loss_cnn = x_real.new_zeros(())

            # 3) Blended loss and step
            loss = gamma * loss_noise + (1.0 - gamma) * loss_cnn
            loss.backward()
            optim.step()

            # tracking
            loss_val = float(loss.detach().item())
            train_loss_sum += loss_val
            train_loss_batches += 1

            l_noise = float(loss_noise.detach().item())
            l_cnn   = float(loss_cnn.detach().item()) if (use_cnn and gamma < 1.0) else 0.0
            loss_ema = loss_val if loss_ema is None else (0.95 * loss_ema + 0.05 * loss_val)
            pbar.set_description(f"loss:{loss_ema:.4f} | noise:{l_noise:.3e} | cnn:{l_cnn:.3e}")

        # Mean train loss this epoch
        mean_train_loss = train_loss_sum / max(train_loss_batches, 1)
        train_losses.append(mean_train_loss)

        # ---- Evaluation on test subset (blended) ----
        mean_test_loss = evaluate_on_test_portion()
        test_losses.append(mean_test_loss)
        print(f"epoch {ep}: train_loss={mean_train_loss:.6f} | test_loss={mean_test_loss:.6f}")

        # ---- Save best model ----
        if save_model and mean_test_loss < best_test_loss:
            best_test_loss = mean_test_loss
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(ddpm.state_dict(), best_path)
            print(f"✔ improved test loss; saved best model to {best_path}")

        # ---- Plot & save loss curves ----
        try:
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(train_losses)+1), train_losses, label="Train (blended)")
            plt.plot(range(1, len(test_losses)+1),  test_losses,  label="Test (blended)")
            plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"Train vs Test Loss (gamma={gamma})")
            plt.legend(); plt.tight_layout()
            loss_curve_path = os.path.join(save_dir, "loss_curve.png")
            plt.savefig(loss_curve_path, dpi=150)
            plt.close()
            print(f"updated loss curve at {loss_curve_path}")
        except Exception as e:
            print(f"warning: failed to plot loss curve ({e})")

    # Optionally save final model state
    if save_model:
        final_path = os.path.join(save_dir, f"model_final.pth")
        torch.save(ddpm.state_dict(), final_path)
        print(f"saved final model at {final_path}")

    # ---- FINAL-EPOCH SUMMARY IMAGE ----
    try:
        out_png = os.path.join(save_dir, "compare_3gens.png")
        compare_waveguides_3gens(
            ddpm,
            test_loader,
            device=device,
            guide_w=2.0,
            n_rows=12,
            save_path=out_png,
            binarize=True,
            thresh=0.5,
        )
    except Exception as e:
        print(f"warning: failed to create final comparison grid ({e})")


In [ ]:
if __name__ == "__main__":
    train_waveguide_ddpm(cnn=cnn, gamma=1, save_dir="./data/diffusion_dispersion_gamma00/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.9, save_dir="./data/diffusion_dispersion_gamma09/")
    train_waveguide_ddpm(cnn=cnn, gamma=0.5, save_dir="./data/diffusion_dispersion_gamma05/")

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def compare_waveguides_3gens_cnn(
    ddpm,
    cnn,                           # NEW: evaluator CNN (x:[B,1,32,32], params:[B,4]) -> [B,2]
    test_loader,
    device,
    guide_w=2.0,
    n_rows=12,
    save_path=None,
    binarize=False,
    thresh=0.5,
    wavelength_index=0,            # which params index encodes wavelength for plotting
    line_alpha=0.95                # line alpha for green curves
):
    """
    For each of the first n_rows items from the test loader, generate 3 waveguides
    conditioned on the same dispersion sequence (cond_seq, params_seq), and display as:

      [Real (red) | Gen1 (black) | Gen2 (black) | Gen3 (black) | Mode vs λ | Weight vs λ]

    Dispersion plots (last 2 columns):
      - Target (dataset) mode/weight sequence: red line
      - CNN-predicted mode/weight for each of the 3 generated waveguides: green lines
      - y-lims: mode ±1 around target, weight ±15 around target
      - grid lines enabled

    test_loader should yield either:
      NEW: (cond:[B,W,2], params:[B,W,4], x_real:[B,1,32,32])
      OLD: (top_pair:[B,2] or [B,4,2], params:[B,4], x_real:[B,1,32,32])
           (old inputs are automatically tiled across W by _ensure_sequences)
    """
    ddpm.eval()
    cnn.eval()

    # Grab one batch
    try:
        cond_or_top, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    # Trim to n_rows
    n = min(n_rows, cond_or_top.shape[0])
    cond_or_top = cond_or_top[:n].to(device)
    params      = params[:n].to(device)
    x_real      = x_real[:n].to(device)

    # Ensure sequences
    cond_seq, params_seq = _ensure_sequences(cond_or_top, params)  # [n,W,2], [n,W,4]
    W = cond_seq.size(1)

    # Repeat each condition 3× (independent noise makes different samples)
    num_gens = 3
    cond_seq_rep   = cond_seq.repeat_interleave(num_gens, dim=0)     # [n*3,W,2]
    params_seq_rep = params_seq.repeat_interleave(num_gens, dim=0)   # [n*3,W,4]

    # Sample all at once
    with torch.no_grad():
        x_gen_all, _ = ddpm.sample(
            n_sample=n * num_gens,
            size=(1, 32, 32),
            device=device,
            cond_seq=cond_seq_rep,
            params_seq=params_seq_rep,
            guide_w=guide_w,
        )  # [n*3,1,32,32]

    # Reshape to [n,3,1,32,32]
    x_gen_all = x_gen_all.view(n, num_gens, 1, 32, 32)

    # To CPU numpy for image display (clamp and optional binarization)
    real_np = x_real.detach().float().cpu().numpy()           # [n,1,32,32]
    gen_np  = x_gen_all.detach().float().cpu().numpy()        # [n,3,1,32,32]

    real_np = np.clip(real_np, 0.0, 1.0)
    gen_np  = np.clip(gen_np,  0.0, 1.0)
    if binarize:
        real_np = (real_np >= thresh).astype(np.float32)
        gen_np  = (gen_np  >= thresh).astype(np.float32)

    # Figure: 6 columns = 4 images + 2 dispersion plots
    cols = 6
    fig_h = max(2, n * 1.3)
    fig, axs = plt.subplots(n, cols, figsize=(cols * 2.4, fig_h), squeeze=False)

    with torch.no_grad():
        for i in range(n):
            # ---------- Image columns ----------
            # Real (red)
            ax = axs[i, 0]
            ax.imshow(real_np[i, 0], cmap="Reds", vmin=0, vmax=1)
            if i == 0: ax.set_title("Real", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

            # Gen 1..3 (black on white)
            for j in range(num_gens):
                ax = axs[i, j + 1]
                ax.imshow(1.0 - gen_np[i, j, 0], cmap="binary_r", vmin=0, vmax=1)
                if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
                ax.set_xticks([]); ax.set_yticks([])

            # ---------- Dispersion columns ----------
            # Target (dataset) sequences
            target_seq = cond_seq[i].detach().cpu().numpy()     # [W,2] -> [:,0]=mode, [:,1]=weight
            target_mode   = target_seq[:, 0]
            target_weight = target_seq[:, 1]

            # Wavelength x-axis from params; if missing/degenerate, fallback to index
            wl = params_seq[i, :, wavelength_index].detach().cpu().numpy()
            if np.allclose(wl, wl[0]):  # constant -> use index
                wl = np.arange(W)

            # CNN-predicted dispersion for each of the 3 generated waveguides, batched
            # x_gen_i: [3,1,32,32] -> repeat across W and stack with params [W,4]
            x_gen_i = x_gen_all[i]                                     # [3,1,32,32]
            x_rep = x_gen_i.unsqueeze(1).repeat(1, W, 1, 1, 1)         # [3,W,1,32,32]
            x_rep = x_rep.view(-1, 1, 32, 32)                          # [3W,1,32,32]

            p_rep = params_seq[i].unsqueeze(0).repeat(3, 1, 1)         # [3,W,4]
            p_rep = p_rep.view(-1, params_seq.shape[-1])               # [3W,4]

            preds = cnn(x_rep.to(device), p_rep.to(device))            # [3W,2]
            preds = preds.view(3, W, 2).detach().cpu().numpy()         # [3,W,2]
            pred_modes   = preds[:, :, 0]                               # [3,W]
            pred_weights = preds[:, :, 1]                               # [3,W]

            # ---- Mode vs λ plot (col 4) ----
            ax_mode = axs[i, 4]
            ax_mode.plot(wl, target_mode, color="red", linewidth=1.8, label="Target")
            for j in range(num_gens):
                ax_mode.plot(wl, pred_modes[j], color="green", alpha=line_alpha, linewidth=1.2,
                             label=("Gen 1" if (i==0 and j==0) else None))
            # y-lims: ±1 around target range
            ymin = float(np.min(target_mode) - 1.0)
            ymax = float(np.max(target_mode) + 1.0)
            if ymin >= ymax:  # degenerate safety
                ymin, ymax = ymin - 1.0, ymax + 1.0
            ax_mode.set_ylim(ymin, ymax)
            ax_mode.grid(True, alpha=0.35, linestyle="--")
            if i == 0:
                ax_mode.set_title("Mode vs λ", fontsize=10)
                ax_mode.legend(loc="best", fontsize=8, frameon=False)
            if i == n - 1:
                ax_mode.set_xlabel("Wavelength", fontsize=9)
            ax_mode.set_ylabel("Mode", fontsize=9)

            # ---- Weight vs λ plot (col 5) ----
            ax_w = axs[i, 5]
            ax_w.plot(wl, target_weight, color="red", linewidth=1.8, label="Target")
            for j in range(num_gens):
                ax_w.plot(wl, pred_weights[j], color="green", alpha=line_alpha, linewidth=1.2,
                          label=("Gen 1" if (i==0 and j==0) else None))
            # y-lims: ±15 around target range
            ymin_w = float(np.min(target_weight) - 15.0)
            ymax_w = float(np.max(target_weight) + 15.0)
            if ymin_w >= ymax_w:  # degenerate safety
                ymin_w, ymax_w = ymin_w - 15.0, ymax_w + 15.0
            ax_w.set_ylim(ymin_w, ymax_w)
            ax_w.grid(True, alpha=0.35, linestyle="--")
            if i == 0:
                ax_w.set_title("Weight vs λ", fontsize=10)
            if i == n - 1:
                ax_w.set_xlabel("Wavelength", fontsize=9)
            ax_w.set_ylabel("Weight", fontsize=9)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()

In [ ]:
h5_path= 'wavelength_grid_dataset.h5'
stats_path = "waveguide_stats_log_norm_above90.npz"
n_epoch=20
batch_size=256
n_T=400
n_feat=128
lrate=1e-4
drop_prob=0.1
betas=(1e-4, 0.02)
ws_test=(0.0, 0.5, 2.0)
save_model=True
test_eval_fraction=0.25   # portion of test loader to evaluate each epoch
save_dir = '/data/diffusion_dispersion_gamma00'

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ---- Data ----
train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# ---- Model ----
unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)
ddpm.load_state_dict(torch.load(f'{save_dir}/best_model.pth', map_location=torch.device(device)))

compare_waveguides_3gens_cnn(
    ddpm=ddpm,
    cnn=cnn,                           # NEW: evaluator CNN (x:[B,1,32,32], params:[B,4]) -> [B,2]
    test_loader=test_loader,
    device=device,
    guide_w=2.0,
    n_rows=12,
    save_path=f'{save_dir}/compare_3gens_cnn.png',
    binarize=False,
    thresh=0.5,
    wavelength_index=0,            # which params index encodes wavelength for plotting
    line_alpha=0.95                # line alpha for green curves
)